In [1]:
# Import necessary modules
from qiskit import QuantumRegister, ClassicalRegister, QuantumCircuit, transpile
from qiskit.circuit.library import CXGate
# from qiskit_ibm_provider import IBMProvider
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
from qiskit.transpiler import CouplingMap
from qiskit.circuit.library import C4XGate
from qiskit_aer import AerSimulator
# from qiskit.providers.fake_provider import FakeCambridge
QiskitRuntimeService.delete_account()

import time
import os
# QiskitRuntimeService.save_account(token="IYxMCq8Fhs5NzG80K3br2EhvbU5WziWlfypyuyr3SELB",channel="ibm_quantum",overwrite=True )
# Set the API key as an environment variable

# Initialize the Qiskit Runtime Service
your_api_key = "V_5F6RZPUb9GB37hSH3CQrQpdFX7WpfP2kek91zjo0Ml"
your_crn = "crn:v1:bluemix:public:quantum-computing:us-east:a/5e194ca79679454da57a2ad2d3e49a9d:749f65fa-d77a-44f6-913a-5a77858ae7aa::"

from qiskit_ibm_runtime import QiskitRuntimeService

QiskitRuntimeService.save_account(
    channel="ibm_cloud",
    token=your_api_key,
    instance="crn:v1:bluemix:public:quantum-computing:us-east:a/5e194ca79679454da57a2ad2d3e49a9d:749f65fa-d77a-44f6-913a-5a77858ae7aa::",
    overwrite=True
)
service = QiskitRuntimeService()
# Specify the backend
backend_name = 'ibm_brisbane'
backend = service.backend(backend_name)

sim =AerSimulator()
# Example quantum circuit
qreg_q = QuantumRegister(5, 'q')
cal_qc = QuantumCircuit(qreg_q)
from qiskit import QuantumCircuit
import numpy as np


def derive_coupling_map_from_circuit(transpiled_circuit):
    derived_coupling_map = set()  # Use a set to avoid duplicate entries

    # Initialize the swap history for each qubit in the circuit
    qubit_swap_history = {qubit: False for qubit in transpiled_circuit.qubits}

    for instruction in transpiled_circuit.data:
        if instruction[0].name == 'swap':
            # If the instruction is a SWAP gate, mark the qubits as involved in a swap
            for qarg in instruction[1]:
                qubit_swap_history[qarg] = True
            continue  # Move to the next instruction

        if len(instruction[1]) == 2:
            # Directly use the qubit objects
            qubits = [qarg for qarg in instruction[1]]

            # Check if any of the qubits were involved in a SWAP
            if not any(qubit_swap_history[qubit] for qubit in qubits):
                # If neither qubit has been swapped, add the connection
                qubit_indices = [transpiled_circuit.find_bit(qarg).index for qarg in qubits]
                derived_coupling_map.add(tuple(sorted(qubit_indices)))

    return derived_coupling_map

In [2]:
from collections import Counter, defaultdict
from qiskit.circuit.random import random_circuit
from qiskit import transpile
import numpy as np
np.random.seed(42)
from qiskit.quantum_info import state_fidelity, partial_trace, DensityMatrix
# from qiskit_aer.primitives import Sampler  # Aer Sampler runs locally
from qiskit_ibm_runtime import Sampler
from qiskit import generate_preset_pass_manager

# Assume backend is defined somewhere earlier in your environment
sampler = Sampler(backend)  # no backend needed, runs on AerSimulator under the hood

def standardize_counts(counts, num_qubits):
    """Convert all keys to little-endian bitstring format used by Qiskit get_counts()."""
    new_counts = {}
    for k, v in counts.items():
        if isinstance(k, int):  # integer key
            bitstring = format(k, f"0{num_qubits}b")[::-1]  # reverse for little-endian
        elif isinstance(k, (list, tuple)):  # tuple or list of bits
            bitstring = "".join(str(b) for b in reversed(k))  # reverse order
        else:  # assume string
            bitstring = k
        new_counts[bitstring] = new_counts.get(bitstring, 0) + v
    return new_counts

def get_ideal_counts(circuit, shots=1000):
    # Ensure circuit has measurements for counts

    
    job = sim.run(circuit, shots=shots)
    result = job.result()
    counts = result.get_counts()
    return counts
def normalize_counts(counts):
    """Normalize counts dictionary to probabilities."""
    total = sum(counts.values())
    return {k: v / total for k, v in counts.items()}

def classical_fidelity(p, q):
    """Calculate classical fidelity (Bhattacharyya coefficient) between two distributions p and q."""
    all_keys = set(p.keys()).union(q.keys())
    fidelity = 0
    for k in all_keys:
        fidelity += np.sqrt(p.get(k, 0) * q.get(k, 0))
    return fidelity

def compute_circuit_fidelity(qc, backend, shots=1000):
    cr = ClassicalRegister(1, name='c')
    qc.add_register(cr)


    # Allowed gates (no "dcx")
    allowed_gates = ["cx", "h", "x", "y", "z", "rx", "ry", "rz", "sx", "u", "id"]

  

    # Convert to only allowed gates (transpile)

    qc = transpile(qc, basis_gates=allowed_gates)

    # Measure qubit 1 into the classical bit 0 of 'c'
    qc.measure(2, cr[0])
    pm = generate_preset_pass_manager(backend=backend, optimization_level=3)
    qc_isa = pm.run(qc)

    # -------------------
    # 1) Run on real backend (or Sampler)
    sampler = Sampler(mode=backend)
    counts_backend = sampler.run([qc_isa], shots=1000).result()[0].data.c.get_counts()
    print("Counts from backend:", counts_backend)
    backend = AerSimulator()
    # -------------------
    # 2) Run same transpiled circuit on statevector simulator for fidelity
    pm = generate_preset_pass_manager(backend=sim, optimization_level=3)
    qc_isa = pm.run(qc)

    # -------------------
    # 1) Run on real backend (or Sampler)
    sampler = Sampler(mode=backend)
    counts_sim = sampler.run([qc_isa], shots=1000).result()[0].data.c.get_counts()
    print("Counts from backend:", counts_sim)

    all_keys = set(counts_backend.keys()).union(set(counts_sim.keys()))
    fidelity = sum(
        (counts_backend.get(k, 0) / shots) ** 0.5 *
        (counts_sim.get(k, 0) / shots) ** 0.5
        for k in all_keys
    ) ** 2

    print("Classical fidelity:", fidelity)
    return fidelity

def compute_transpiler_fidelity_expression(transpiled_circuit, derived_coupling_map):
    """
    Build a symbolic fidelity LHS expression from the transpiled circuit.
    Does not use backend properties or calculate numeric RHS.
    """
    gate_counts = defaultdict(int)
    readout_qubits = set()

    for inst, qargs, _ in transpiled_circuit.data:
        if not qargs:
            continue  # skip barriers or non-qubit ops
        q_indices = tuple(transpiled_circuit.find_bit(qarg).index for qarg in qargs)
        sorted_indices = tuple(sorted(q_indices))

        # Only include 2Q gates present in derived coupling map
        if len(q_indices) == 2 and sorted_indices not in derived_coupling_map:
            continue

        if inst.name == "measure":
            readout_qubits.add(q_indices[0])
        else:
            gate_counts[(inst.name, sorted_indices)] += 1

    symbolic_terms = []
    for (gname, qubits), count in gate_counts.items():
        symbolic_terms.append(f"(1 - ε_{gname}{qubits})^{count}")

    for q in sorted(readout_qubits):
        symbolic_terms.append(f"(1 - ε_ro_q{q})")

    return " * ".join(symbolic_terms)

def build_fidelity_equation_with_rhs(transpiled_circuit, derived_coupling_map, shots=1000):
    lhs_expr = compute_transpiler_fidelity_expression(transpiled_circuit, derived_coupling_map)
    rhs_fid = compute_circuit_fidelity(transpiled_circuit, backend, shots)
    rhs_value = f"{rhs_fid:.6f}"
    return f"{lhs_expr} = {rhs_value}"

equation_list = []

# for _ in range(10):
for i in range(10):
    qc = random_circuit(10, 2, max_operands=2, seed=42+i)  # change seed to vary circuit
    big_rc = derive_coupling_map_from_circuit(qc)
    symbolic = build_fidelity_equation_with_rhs(qc, big_rc, shots=1000)
    equation_list.append(symbolic)

print("\n".join(equation_list))




C:\Users\rupsh\AppData\Local\Temp\ipykernel_5944\2195800568.py:49: DeprecationWarning: Treating CircuitInstruction as an iterable is deprecated legacy behavior since Qiskit 1.2, and will be removed in Qiskit 3.0. Instead, use the `operation`, `qubits` and `clbits` named attributes.
  if instruction[0].name == 'swap':
C:\Users\rupsh\AppData\Local\Temp\ipykernel_5944\2195800568.py:55: DeprecationWarning: Treating CircuitInstruction as an iterable is deprecated legacy behavior since Qiskit 1.2, and will be removed in Qiskit 3.0. Instead, use the `operation`, `qubits` and `clbits` named attributes.
  if len(instruction[1]) == 2:
C:\Users\rupsh\AppData\Local\Temp\ipykernel_5944\2195800568.py:57: DeprecationWarning: Treating CircuitInstruction as an iterable is deprecated legacy behavior since Qiskit 1.2, and will be removed in Qiskit 3.0. Instead, use the `operation`, `qubits` and `clbits` named attributes.
  qubits = [qarg for qarg in instruction[1]]
C:\Users\rupsh\AppData\Local\Temp\ipyke

Counts from backend: {'0': 493, '1': 507}
Counts from backend: {'0': 500, '1': 500}
Classical fidelity: 0.9999509975987648
Counts from backend: {'0': 979, '1': 21}
Counts from backend: {'0': 1000}
Classical fidelity: 0.9789999999999999
Counts from backend: {'0': 985, '1': 15}
Counts from backend: {'0': 1000}
Classical fidelity: 0.9850000000000001
Counts from backend: {'0': 406, '1': 594}
Counts from backend: {'1': 573, '0': 427}
Classical fidelity: 0.9995463421951689
Counts from backend: {'0': 997, '1': 3}
Counts from backend: {'0': 1000}
Classical fidelity: 0.997
Counts from backend: {'1': 610, '0': 390}
Counts from backend: {'1': 618, '0': 382}
Classical fidelity: 0.9999324903409479
Counts from backend: {'1': 986, '0': 14}
Counts from backend: {'1': 1000}
Classical fidelity: 0.986
Counts from backend: {'0': 197, '1': 803}
Counts from backend: {'1': 946, '0': 54}
Classical fidelity: 0.9500650880337292
Counts from backend: {'0': 392, '1': 608}
Counts from backend: {'1': 643, '0': 357}


RequestsApiError: 'HTTPSConnectionPool(host=\'us-east.quantum-computing.cloud.ibm.com\', port=443): Max retries exceeded with url: /jobs/d2d7u2eaa61s73ee5hf0?exclude_params=true (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001F09B66FC40>: Failed to resolve \'us-east.quantum-computing.cloud.ibm.com\' ([Errno 11001] getaddrinfo failed)"))'